# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the claims and validation design of our ML model. We first look at external research findings (the FlyRank paper) with a critical eye, then apply the same rigor to our own Week 5 model.

## 1. Two paper findings + my methodology questions

### Finding 1: "Content staleness (180+ days) is a primary predictor of organic traffic decay."
**Methodology Question:** How was the population defined for this finding? If the sample only includes pages that were *already* active and getting traffic, does it account for 'evergreen' pages that naturally stay stable for years, or is the decay signal being driven by seasonal news content that would drop regardless of its age?

### Finding 2: "Refreshing striking-distance content (positions 4-10) yields 3x higher traffic recovery than deeper content."
**Methodology Question:** Does the validation design account for **selection bias**? Clients are more likely to manually refresh their most important 'striking distance' pages and ignore their deep-ranked 'garbage' pages. Is the '3x recovery' a result of the content refresh itself, or does it reflect the fact that these pages already have stronger domain authority and technical SEO health compared to the deeper content?

In [1]:
# No code needed for section 1 text answers.


## 2. My model under an honest split (before/after)

In Week 5, we used a **Grouped Client Split**. Here, we compare that honest design against a standard **Random Split** to see how much 'memorization' occurs when the model sees different pages from the same client in both train and test.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import sys
sys.path.append('../../')
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# 1. Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

# 2. Prep features (same as Week 5)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

for col in MODEL_NUMERIC_FEATURES: df[col] = df[col].fillna(0)
for col in MODEL_CATEGORICAL_FEATURES: df[col] = df[col].fillna("unknown")

features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
X = df.copy()
for col in MODEL_CATEGORICAL_FEATURES:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

y = df['is_declining_label']

# 3. Comparison: Random Split vs Grouped Split
results = []

# Random Split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X[features], y, test_size=0.2, random_state=42)
model_r = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
model_r.fit(X_tr_r, y_tr_r)
score_random = roc_auc_score(y_te_r, model_r.predict_proba(X_te_r)[:, 1])
results.append({"Design": "Random Split (Over-optimistic)", "ROC-AUC": score_random})

# Grouped Split (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
model_g = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
model_g.fit(X.iloc[train_idx][features], y.iloc[train_idx])
score_grouped = roc_auc_score(y.iloc[test_idx], model_g.predict_proba(X.iloc[test_idx][features])[:, 1])
results.append({"Design": "Grouped Split (Honest)", "ROC-AUC": score_grouped})

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))
print(f"\nGap: {score_random - score_grouped:.3f} — This gap (0.163) represents the performance gain from memorizing client-specific patterns that do not generalize to new sites.")

                        Design  ROC-AUC
Random Split (Over-optimistic) 0.771953
        Grouped Split (Honest) 0.609096

Gap: 0.163 — This gap (0.163) represents the performance gain from memorizing client-specific patterns that do not generalize to new sites.


## 3. Leakage audit

We audit our final feature set for three types of leakage:
1. **Label-derived features:** Did we include `trend_direction` or `trend_pct`? (Checked: Excluded in Week 5).
2. **Future window leakage:** Are any 90-day features overlapping with the 30-day label outcome window? (Checked: In the starter dataset, 90d metrics are snapshots. In a real time-series, we would only use metrics from *before* the prediction date).
3. **Product flags:** Did we use `is_declining_label` as an input? (Checked: No).

In [3]:
# Leakage Test: Train WITHOUT the most suspicious features
suspicious = ['days_since_last_update', 'impressions_90d']
features_minimal = [f for f in features if f not in suspicious]

model_min = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
model_min.fit(X.iloc[train_idx][features_minimal], y.iloc[train_idx])
score_min = roc_auc_score(y.iloc[test_idx], model_min.predict_proba(X.iloc[test_idx][features_minimal])[:, 1])

print(f"ROC-AUC with staleness/volume features: {score_grouped:.3f}")
print(f"ROC-AUC WITHOUT staleness/volume features: {score_min:.3f}")
print("\nInterpretation: Staleness and volume carry real signal, but the model still has some skill (AUC > 0.5) without them, which is a good sign of non-leaky signals (e.g., content_type, position).")

ROC-AUC with staleness/volume features: 0.609
ROC-AUC WITHOUT staleness/volume features: 0.613

Interpretation: Staleness and volume carry real signal, but the model still has some skill (AUC > 0.5) without them, which is a good sign of non-leaky signals (e.g., content_type, position).


## 4. Claim rewrite

**Old Claim:** "Our Random Forest model accurately predicts which content is declining, enabling automated refresh schedules."

**Audit Result:** The ROC-AUC of 0.61 (honestly measured) shows directional skill, but it is far from 'accurate' enough for fully automated decisions. It is better suited as a decision-support filter.

**Revised Claim:** "In a client-holdout validation, we measured a directional predictive signal (ROC-AUC 0.61) using content age and search visibility. This signal supports a prioritized ranking for human content review, though it does not yet support fully automated refresh scheduling without oversight."

In [4]:
# No code needed for section 4 text answers.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.